# Step 6 - Parametric dependence of the X-line and comparison with the models

Rotation rate of the X-line with the IMF clock angle and shift rate with the dipole tilt, for the BL X-line (step 5) and for the model X-lines (DXL: dominant X-line, MRL: maximum reconnection rate line).

Inputs: `{OUT}/xlines/DeltaVz_*.pkl` (step 5), `{MODEL_LINES}/DXL_cla_*_coa_75_tilt_*.pkl` and `{MODEL_LINES}/MRL_cla*_tilt*.pkl`.

Copied from Article3.ipynb (cells 10 and 12, disabled with a docstring, are not copied).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # makes the blxline package importable
from blxline.paths import OUT, MODEL_LINES  # all outputs of this repo are written under OUT (never over the original files)

# Plot all curves

In [ ]:
# [from Article3.ipynb, cell 1]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from spok.plot import planet_env
from spok.models.planetary import Magnetosheath


In [ ]:
# [from Article3.ipynb, cell 2]
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree


def label_pieces(y, z, max_gap=2.):
    """Label of the piece each point belongs to (-1 for NaN points)."""
    y, z = np.asarray(y, dtype=float), np.asarray(z, dtype=float)
    labels = np.full(y.shape, -1)
    valid = np.isfinite(y) & np.isfinite(z)
    pts = np.array([y[valid], z[valid]]).T
    if len(pts) == 0:
        return labels
    pairs = cKDTree(pts).query_pairs(max_gap, output_type='ndarray')
    graph = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(pts), len(pts)))
    labels[valid] = connected_components(graph, directed=False)[1]
    return labels


def piece_extent(y, z):
    """Size of a piece: diagonal of its bounding box."""
    return np.hypot(np.ptp(y), np.ptp(z))


def keep_main_part(y, z, max_gap=2., min_fraction=None, return_mask=False):
    """Remove the pieces of the line (y, z) that are far from its main part.
    y, z : positions of the points of the line (the order is kept)
    max_gap : points closer than max_gap are in the same piece. Should be larger than the
        gaps you accept in a line, and smaller than the distance of the pieces to remove.
    min_fraction : if given, also keep the pieces whose extent is at least min_fraction times
        the extent of the main piece.
    Returns the cleaned y, z (and the boolean mask of the kept points if return_mask).
    """
    y, z = np.asarray(y, dtype=float), np.asarray(z, dtype=float)
    labels = label_pieces(y, z, max_gap=max_gap)
    pieces = np.unique(labels[labels >= 0])
    if len(pieces) == 0:
        mask = np.zeros(y.shape, dtype=bool)
    else:
        extents = np.array([piece_extent(y[labels == p], z[labels == p]) for p in pieces])
        if min_fraction is None:
            kept = pieces[[np.argmax(extents)]]
        else:
            kept = pieces[extents >= min_fraction * extents.max()]
        mask = np.isin(labels, kept)
    if return_mask:
        return y[mask], z[mask], mask
    return y[mask], z[mask]

In [ ]:
# [from Article3.ipynb, cell 3]
clas = [120,125,130,135,140,145,150,155,160,165,170,175,180]
tilts = [-30,-25,-20,-15,-10,-5, 0,5,10,15,20,25,30]


## deltaVz = 0

In [ ]:
# [from Article3.ipynb, cell 5]
msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
ncols = 5
nrows = int(np.ceil(len(clas)/ncols))
fig, ax = plt.subplots(ncols=ncols,nrows=nrows,figsize=(ncols*5,nrows*5))

count = 0
for cla in clas:
    for tilt in tilts:
        res = pd.read_pickle(f'{OUT}/xlines/DeltaVz_cla{cla}_tilt{tilt}.pkl')
        ax[count//ncols, count%ncols].plot(*keep_main_part(res['Yxline'],res['Zxline']),label=r'$\Psi$'+f'={tilt}°')
    ax[count//ncols, count%ncols].legend()
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([ax[count//ncols, count%ncols]]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))
    ax[count//ncols, count%ncols].set_xlim(-17,17)
    ax[count//ncols, count%ncols].set_ylim(-17,17)
    ax[count//ncols, count%ncols].set_aspect('equal')
    ax[count//ncols, count%ncols].axvline(0,linestyle='--',color='k')
    ax[count//ncols, count%ncols].axhline(0,linestyle='--',color='k')
    ax[count//ncols, count%ncols].set_title(r'$\theta_{cl}$'+f'={cla}°')
    count += 1
#fig.tight_layout()

In [ ]:
# [from Article3.ipynb, cell 6]
msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
ncols = 5
nrows = int(np.ceil(len(clas)/ncols))
fig, ax = plt.subplots(ncols=ncols,nrows=nrows,figsize=(ncols*5,nrows*5))

count = 0
for cla in clas:
    for tilt in tilts:
        res = pd.read_pickle(f'{MODEL_LINES}/DXL_cla_{cla}_coa_75_tilt_{tilt}.pkl')
        ax[count//ncols, count%ncols].plot(res['y'],res['z'],label=r'$\Psi$'+f'={tilt}°')
    ax[count//ncols, count%ncols].legend()
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([ax[count//ncols, count%ncols]]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))
    ax[count//ncols, count%ncols].set_xlim(-17,17)
    ax[count//ncols, count%ncols].set_ylim(-17,17)
    ax[count//ncols, count%ncols].set_aspect('equal')
    ax[count//ncols, count%ncols].axvline(0,linestyle='--',color='k')
    ax[count//ncols, count%ncols].axhline(0,linestyle='--',color='k')
    ax[count//ncols, count%ncols].set_title(r'$\theta_{cl}$'+f'={cla}°')
    count += 1
#fig.tight_layout()

In [ ]:
# [from Article3.ipynb, cell 7]
msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
ncols = 5
nrows = int(np.ceil(len(clas)/ncols))
fig, ax = plt.subplots(ncols=ncols,nrows=nrows,figsize=(ncols*5,nrows*5))

count = 0
for cla in clas:
    for tilt in tilts:
        res = pd.read_pickle(f'{MODEL_LINES}/MRL_cla{cla}_tilt{tilt}.pkl')
        ax[count//ncols, count%ncols].plot(res['y'],res['z'],label=r'$\Psi$'+f'={tilt}°')
    ax[count//ncols, count%ncols].legend()
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([ax[count//ncols, count%ncols]]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))
    ax[count//ncols, count%ncols].set_xlim(-17,17)
    ax[count//ncols, count%ncols].set_ylim(-17,17)
    ax[count//ncols, count%ncols].set_aspect('equal')
    ax[count//ncols, count%ncols].axvline(0,linestyle='--',color='k')
    ax[count//ncols, count%ncols].axhline(0,linestyle='--',color='k')
    ax[count//ncols, count%ncols].set_title(r'$\theta_{cl}$'+f'={cla}°')
    count += 1
#fig.tight_layout()

![image.png](attachment:1cf14d19-bb11-4fc5-bb78-a24cd7a92f6c.png)

In [ ]:
# [from Article3.ipynb, cell 9]
msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
ncols = 5
nrows = int(np.ceil(len(tilts)/ncols))
fig, ax = plt.subplots(ncols=ncols,nrows=nrows,figsize=(ncols*5,nrows*5))

count = 0
for tilt in tilts:
    for cla in clas:
        res = pd.read_pickle(f'{OUT}/xlines/DeltaVz_cla{cla}_tilt{tilt}.pkl')
        y,z = res['Yxline'],res['Zxline']
        y,z = keep_main_part(y,z,max_gap=3)
        ax[count//ncols, count%ncols].scatter(y,z,label=r'$\theta_{cl}$'+f'={cla}°',s=3)
    ax[count//ncols, count%ncols].legend()
    _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([ax[count//ncols, count%ncols]]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))
    ax[count//ncols, count%ncols].set_xlim(-17,17)
    ax[count//ncols, count%ncols].set_ylim(-17,17)
    ax[count//ncols, count%ncols].set_aspect('equal')
    ax[count//ncols, count%ncols].axvline(0,linestyle='--',color='k')
    ax[count//ncols, count%ncols].axhline(0,linestyle='--',color='k')
    ax[count//ncols, count%ncols].set_title(r'$\Psi$'+f'={tilt}°')
    count += 1
#fig.tight_layout()

### Check individual problems and how to fix them
- $\Psi=-25°$, $\theta=140°$ --> x<0
- $\Psi=-20°$, $\theta=135,140,145°$ --> x<0
- $\Psi=-15°$, $\theta=145,155°$ --> x<0 à peu près
- $\Psi=-10°$, $\theta=130,135,140,145°$ --> trous à droite?
- $\Psi=-5°$, $\theta=135,140,145°$ --> droite par morceaux pas bien rangés (donc scatter plutôt que plot règle ce pb)
- $\Psi=5°$, $\theta=145°$ --> droite par morceaux pas bien rangés (donc scatter plutôt que plot règle ce pb)
- $\Psi=10°$, $\theta=125,130°$ --> trous?
- $\Psi=25°$, $\theta=165,170,175,180°$ --> droite par morceaux pas bien rangés (donc scatter plutôt que plot règle ce pb)

Actually when plotting with scatter instead of plot, I find that the only problem still here are the points at x<0! (which is like 1 point on 5 five maps so I don't care, it will not mess up with the computaion of the slopes.

Actually when doing that I find that the flattening of the curves is not due to missing points, and the continuous by segments is not a problem when using scatter instead of plot. the only problem left is the point at x<0, but it's like 1 point in 5 maps so I don't care, it will not mess up the computation of the slopes!

## Computing the rotation rate

In [ ]:
# [from Article3.ipynb, cell 16]
from scipy.ndimage import gaussian_filter
from scipy.stats import linregress

def find_rate_rotation(clas, ys, zs, **kwargs):
    fig, ax = plt.subplots(ncols=2, figsize=(13,7))
    alphas, errors = [], []
    for i,cla in enumerate(clas):
        y,z=ys[i], zs[i]
        # To remove false isolated points in the delta Vz = 0 maps
        # Should be useless but also have no negative impact for other curves
        
        #y,z = y[abs(z)<9], z[abs(z)<9]
        
        # To remove the artificial horizontal blocks 
        #(due to the fact that the points are grid points so there are discrete values of z that can be taken)
        z = gaussian_filter(z,5)  
        # Checking that the curve is increasing (ie central part of the curve, for eastward IMF only!!!)
        
        '''
        cond = (z[1:]-z[:-1])>-0.0001  
        y,z = y[:-1][cond],z[:-1][cond]
        '''
        # For possible residuals of COA or tilt that would shift the line, to remove the shift
        z = z-z[np.argmin(abs(y))]
        # To remove the non linear part of certain curves (such as current and rate at high latitudes)
        r = np.sqrt(y**2+z**2)
        y,z = y[r<kwargs.get('rmax',15)],z[r<kwargs.get('rmax',15)]
        
        ax[0].scatter(y,z,label=f'{round(cla,2)}', s=2)
        # Theoretical curve (compatible with bisector at the subsolar point)
        ax[0].plot(np.linspace(-15,15,50)*np.sin(cla*np.pi/180/2), np.linspace(-15,15,50)*np.cos(cla*np.pi/180/2))
        alpha = np.arctan(z/y)*180/np.pi
        # Remove the possible portions of the line that woule not fall on the upper right or lower left quadrants
        alpha = alpha[alpha>=0]
        alphas += [np.nanmedian(alpha)]
        errors += [np.nanstd(alpha)]
    ax[0].legend(loc='upper left', bbox_to_anchor=(0, 1, 0.5, 0.5))
    ax[0].set_aspect('equal')
    
    # Slope 
    ax[1].errorbar(180-clas, alphas, yerr=errors)
    # Theoretical curve (compatible with bisector at the subsolar point)
    ax[1].plot(np.linspace(0,180-np.min(clas),10),np.linspace(0,180-np.min(clas),10)/2,linestyle='--',color='grey')
    ax[1].set_aspect('equal')
    
    # Mean slope
    rate = (alphas[1:]-alphas[0])/(180-clas[1:])
    rate[abs(rate)>30]=np.nan
    return linregress(180-clas,alphas).slope, linregress(180-clas,alphas).stderr

In [ ]:
# [from Article3.ipynb, cell 17]
values, stds = [],[]

clas = np.array(clas)

for tilt in tilts:
    ys, zs = [],[]
    for cla in clas:
        res = pd.read_pickle(f'{OUT}/xlines/DeltaVz_cla{cla}_tilt{tilt}.pkl')
        y,z = res['Yxline'],res['Zxline']
        y,z = keep_main_part(y,z,max_gap=3)
        ys += [y]
        zs += [z]
    ys = [np.array(val) for val in ys]
    zs = [np.array(val) for val in zs]
    val,std = find_rate_rotation(clas, ys, zs)
    values += [val]
    stds += [std]

In [ ]:
# [from Article3.ipynb, cell 18]
plt.errorbar(tilts,values,yerr=stds)

In [ ]:
# [from Article3.ipynb, cell 19]
clas = np.array(clas)

ys, zs = [],[]
for cla in clas:
    res = pd.read_pickle(f'{OUT}/xlines/DeltaVz_cla{cla}.pkl')
    y,z = res['Yxline'],res['Zxline']
    y,z = keep_main_part(y,z,max_gap=3)
    ys += [y]
    zs += [z]
ys = [np.array(val) for val in ys]
zs = [np.array(val) for val in zs]
val,std = find_rate_rotation(clas, ys, zs)
val, std

In [ ]:
# [from Article3.ipynb, cell 20]
values, stds = [],[]

clas = np.array(clas)

for tilt in tilts:
    ys, zs = [],[]
    for cla in clas:
        res = pd.read_pickle(f'{MODEL_LINES}/DXL_cla_{cla}_coa_75_tilt_{tilt}.pkl')
        y,z = res['y'],res['z']
        ys += [y]
        zs += [z]
    ys = [np.array(val) for val in ys]
    zs = [np.array(val) for val in zs]
    val,std = find_rate_rotation(clas, ys, zs)
    values += [val]
    stds += [std]
    print(f'Done for tilt={tilt}°')
    

In [ ]:
# [from Article3.ipynb, cell 21]
plt.errorbar(tilts,values,yerr=stds)

In [ ]:
# [from Article3.ipynb, cell 22]
values, stds = [],[]

clas = np.array(clas)

for tilt in tilts:
    ys, zs = [],[]
    for cla in clas:
        res = pd.read_pickle(f'{MODEL_LINES}/MRL_cla{cla}_tilt{tilt}.pkl')
        y,z = res['y'],res['z']
        ys += [y]
        zs += [z]
    ys = [np.array(val) for val in ys]
    zs = [np.array(val) for val in zs]
    val,std = find_rate_rotation(clas, ys, zs)
    values += [val]
    stds += [std]
    

In [ ]:
# [from Article3.ipynb, cell 23]
plt.errorbar(tilts,values,yerr=stds)

## "Semi"-conclusion
De gauche à droite : data, DXL, max rate, le rotation rate en ordonnée en fonction du tilt en abcisse

![image.png](attachment:cca5b6ff-bef5-47a5-9048-96d945709b8c.png)
![image.png](attachment:b00b6385-cba9-43ed-aa6a-50013bd68400.png)
![image.png](attachment:ba3b0a62-633c-48a7-b851-5b88b78257de.png)

- Conclure que la valeur du rate est la même pour tous les tilts est pas du tout évidente, d'autant plus que je pense que les tilts proche de 0 sont fucked up par le biais de distributiond es tilt qui applatit les coubres en région subsolaire. En effet, quand on slice pas en tilt mais juste en clock, on a un rotation rate de 0.58.
- Je crois pas qu'un modèle décrive mieux le comportement des lignes que l'autre

## Computing the shift rate

In [ ]:
# [from Article3.ipynb, cell 26]
from scipy.ndimage import gaussian_filter
from scipy.stats import linregress
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def find_rate_shift(tilts, ys, zs, y0=0, **kwargs):
    fig, ax = plt.subplots(figsize=(5,3))
    shifts = []
    for i,tilt in enumerate(tilts):
        y,z=ys[i], zs[i]
        z = z[abs(y-y0)<0.5]
        shifts += [np.nanmedian(z)]
    # Slope 
    ax.errorbar(tilts, shifts)
    ax.set_aspect('equal')
    return linregress(tilts, np.array(shifts)).slope, linregress(tilts, np.array(shifts)).stderr

def find_rate_shift_average(tilts, ys, zs, ymin=-10, ymax=10, **kwargs):
    rates = []
    for y0 in np.linspace(ymin, ymax, 10):
        shifts = []
        for i,tilt in enumerate(tilts):
            y,z=ys[i], zs[i]
            z = z[abs(y-y0)<0.5]
            shifts += [np.nanmedian(z)]
        # Slope 
        rates += [linregress(tilts, np.array(shifts)).slope]
    rates = np.array(rates)
    return np.nanmean(rates), np.nanstd(rates)

In [ ]:
# [from Article3.ipynb, cell 27]
tilts = np.array(tilts)

values0_BL, stds0_BL = [],[]
values4_BL, stds4_BL = [],[]
values_avg_BL, stds_avg_BL = [],[]

for cla in clas:
    ys, zs = [],[]
    for tilt in tilts:
        res = pd.read_pickle(f'{OUT}/xlines/DeltaVz_cla{cla}_tilt{tilt}.pkl')
        y,z = res['Yxline'],res['Zxline']
        y,z = keep_main_part(y,z,max_gap=3)
        ys += [y]
        zs += [z]
    ys = [np.array(val) for val in ys]
    zs = [np.array(val) for val in zs]
    
    val, std = find_rate_shift(tilts, ys, zs, y0=0)
    values0_BL += [val]
    stds0_BL += [std]
    val, std = find_rate_shift(tilts, ys, zs, y0=4)
    values4_BL += [val]
    stds4_BL += [std]
    val, std = find_rate_shift_average(tilts, ys, zs)
    values_avg_BL += [val]
    stds_avg_BL += [std]

In [ ]:
# [from Article3.ipynb, cell 28]
tilts = np.array(tilts)

values0_DXL, stds0_DXL = [],[]
values4_DXL, stds4_DXL = [],[]
values_avg_DXL, stds_avg_DXL = [],[]

for cla in clas:
    ys, zs = [],[]
    for tilt in tilts:
        res = pd.read_pickle(f'{MODEL_LINES}/DXL_cla_{cla}_coa_75_tilt_{tilt}.pkl')
        y,z = res['y'],res['z']
        ys += [y]
        zs += [z]
    ys = [np.array(val) for val in ys]
    zs = [np.array(val) for val in zs]
    
    val, std = find_rate_shift(tilts, ys, zs, y0=0)
    values0_DXL += [val]
    stds0_DXL += [std]
    val, std = find_rate_shift(tilts, ys, zs, y0=4)
    values4_DXL += [val]
    stds4_DXL += [std]
    val, std = find_rate_shift_average(tilts, ys, zs)
    values_avg_DXL += [val]
    stds_avg_DXL += [std]

In [ ]:
# [from Article3.ipynb, cell 29]
tilts = np.array(tilts)

values0_MRL, stds0_MRL = [],[]
values4_MRL, stds4_MRL = [],[]
values_avg_MRL, stds_avg_MRL = [],[]

for cla in clas:
    ys, zs = [],[]
    for tilt in tilts:
        res = pd.read_pickle(f'{MODEL_LINES}/MRL_cla{cla}_tilt{tilt}.pkl')
        y,z = res['y'],res['z']
        ys += [y]
        zs += [z]
    ys = [np.array(val) for val in ys]
    zs = [np.array(val) for val in zs]
    
    val, std = find_rate_shift(tilts, ys, zs, y0=0)
    values0_MRL += [val]
    stds0_MRL += [std]
    val, std = find_rate_shift(tilts, ys, zs, y0=4)
    values4_MRL += [val]
    stds4_MRL += [std]
    val, std = find_rate_shift_average(tilts, ys, zs)
    values_avg_MRL += [val]
    stds_avg_MRL += [std]

In [ ]:
# [from Article3.ipynb, cell 30]
fig, ax = plt.subplots(ncols=3,figsize=(15,5))
ax[0].errorbar(clas,values0_BL,yerr=stds0_BL, label='BL')
ax[0].errorbar(clas,values0_DXL,yerr=stds0_DXL, label='DXL')
ax[0].errorbar(clas,values0_MRL,yerr=stds0_MRL,label='MRL')
ax[0].legend()

ax[1].errorbar(clas,values4_BL,yerr=stds4_BL, label='BL')
ax[1].errorbar(clas,values4_DXL,yerr=stds4_DXL, label='DXL')
ax[1].errorbar(clas,values4_MRL,yerr=stds4_MRL,label='MRL')
ax[1].legend()

ax[2].errorbar(clas,values_avg_BL,yerr=stds_avg_BL, label='BL')
ax[2].errorbar(clas,values_avg_DXL,yerr=stds_avg_DXL, label='DXL')
ax[2].errorbar(clas,values_avg_MRL,yerr=stds_avg_MRL,label='MRL')
ax[2].legend()

# Comparaison position directe

In [ ]:
# [from Article3.ipynb, cell 32]
msh = Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')
nb_plots = len(tilts)*len(clas)
ncols = 7
nrows = int(np.ceil(nb_plots/ncols))
fig, ax = plt.subplots(ncols=ncols,nrows=nrows,figsize=(ncols*3,nrows*3))

count = 0
for tilt in tilts:
    for cla in clas:
        res = pd.read_pickle(f'{OUT}/xlines/DeltaVz_cla{cla}_tilt{tilt}.pkl')
        y,z = res['Yxline'],res['Zxline']
        y,z = keep_main_part(y,z,max_gap=3)
        ax[count//ncols, count%ncols].scatter(y,z,label='BL',s=3)
        res = pd.read_pickle(f'{MODEL_LINES}/DXL_cla_{cla}_coa_75_tilt_{tilt}.pkl')
        ax[count//ncols, count%ncols].plot(res['y'],res['z'],label='DXL')
        res = pd.read_pickle(f'{MODEL_LINES}/MRL_cla{cla}_tilt{tilt}.pkl')
        ax[count//ncols, count%ncols].plot(res['y'],res['z'],label='MRL')   
        
        ax[count//ncols, count%ncols].set_title(r'$\theta_{cl}$'+f'={cla}°, '+r'$\Psi=$'+f'{tilt}°')
        ax[count//ncols, count%ncols].legend()
        _, _ = planet_env.layout_earth_env(msh, figure=fig, axes=np.array([ax[count//ncols, count%ncols]]),x_slice=0, x_lim=(-16,16),y_lim=(-16,16),z_lim=(-16,16))
        ax[count//ncols, count%ncols].set_xlim(-17,17)
        ax[count//ncols, count%ncols].set_ylim(-17,17)
        ax[count//ncols, count%ncols].set_aspect('equal')
        ax[count//ncols, count%ncols].axvline(0,linestyle='--',color='k')
        ax[count//ncols, count%ncols].axhline(0,linestyle='--',color='k')
        count += 1
#fig.tight_layout()